# 02 Data Cleaning

Objective: clean price, missing values, booleans, dates, outliers and create a clean analytical dataset.

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data_cleaning import clean_price_column, standardize_boolean
from src.feature_engineering import create_basic_features, create_classification_target

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

listings = pd.read_csv(DATA_RAW / "listings.csv.gz")

df = listings.copy()
df.shape

(15584, 79)

# Data Cleaning

This notebook transforms the raw Airbnb listings dataset into a cleaned, analysis-ready and model-ready dataset. The main cleaning tasks include price conversion, missing-value handling, outlier filtering, date conversion, Boolean transformation and feature engineering.

In [42]:
columns_to_drop = [
    "neighbourhood_group_cleansed",
    "calendar_updated"
]

df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

df.shape

(14234, 84)

Two columns were removed because they were completely empty: `neighbourhood_group_cleansed` and `calendar_updated`. These variables provide no analytical or predictive value.

In [43]:
df["price"] = clean_price_column(df["price"])

missing_price_before = df["price"].isna().sum()
missing_price_rate = df["price"].isna().mean()

missing_price_before, missing_price_rate.round(3)

(np.int64(0), np.float64(0.0))

In [44]:
df = df[df["price"].notna()].copy()

df.shape

(14234, 84)

The `price` column was converted from text format into numeric format by removing currency symbols and commas. Rows with missing prices were removed because price is the target variable for the regression model.

In [45]:
price_upper_limit = df["price"].quantile(0.99)

price_upper_limit.round(3)

np.float64(399.34)

In [46]:
df = df[df["price"] <= price_upper_limit].copy()

df["price"].describe()

count    14091.000000
mean        94.618196
std         61.731444
min          9.000000
25%         54.000000
50%         76.000000
75%        114.000000
max        398.000000
Name: price, dtype: float64

Extreme price values above the 99th percentile were removed from the model-ready dataset. This approach is less restrictive than the IQR rule and preserves realistic premium listings while excluding extreme observations that could distort machine learning models.

In [47]:
boolean_columns = [
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "instant_bookable",
    "has_availability"
]

for col in boolean_columns:
    if col in df.columns:
        df[col] = standardize_boolean(df[col])

df[boolean_columns].head()

,host_is_superhost,host_has_profile_pic,host_identity_verified,instant_bookable,has_availability
1,0.0,1.0,1.0,0,1.0
2,0.0,1.0,1.0,1,1.0
3,1.0,1.0,1.0,0,1.0
5,0.0,1.0,1.0,0,1.0
6,0.0,1.0,1.0,0,1.0


In [48]:
# clean percentage columns
percentage_columns = [
    "host_response_rate",
    "host_acceptance_rate"
]

for col in percentage_columns:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace("%", "", regex=False)
            .replace("nan", np.nan)
            .astype(float)
        )

df[percentage_columns].describe()


,host_response_rate,host_acceptance_rate
count,12712.000000,13280.000000
mean,97.340072,94.968072
std,11.673807,16.215852
min,0.000000,0.000000
25%,100.000000,99.000000
50%,100.000000,100.000000
75%,100.000000,100.000000
max,100.000000,100.000000


In [49]:
# convert date columns to datetime
date_columns = [
    "last_scraped",
    "host_since",
    "first_review",
    "last_review",
    "calendar_last_scraped"
]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

df[date_columns].dtypes

last_scraped             datetime64[ns]
host_since               datetime64[ns]
first_review             datetime64[ns]
last_review              datetime64[ns]
calendar_last_scraped    datetime64[ns]
dtype: object

In [50]:
# feature engineering
df = create_basic_features(df)
df = create_classification_target(df, "price")

df[["price", "log_price", "amenities_count", "availability_rate", "high_price_listing"]].head()

,price,log_price,amenities_count,availability_rate,high_price_listing
1,45.0,3.828641,25,0.736986,0
2,160.0,5.081404,32,0.197260,1
3,50.0,3.931826,50,0.147945,0
5,70.0,4.262680,14,0.950685,0
6,57.0,4.060443,24,0.000000,0


In [51]:
# create 'has reviews' feature
df["has_reviews"] = (df["number_of_reviews"] > 0).astype(int)

df["has_reviews"].value_counts()

has_reviews
1    12187
0     1904
Name: count, dtype: int64

A binary variable, `has_reviews`, was created to distinguish listings with at least one review from listings with no reviews. This is useful because missing review scores usually indicate absence of reviews rather than random missing data.

In [52]:
selected_columns = [
    "id",
    "name",
    "host_id",
    "host_since",
    "host_is_superhost",
    "host_response_rate",
    "host_acceptance_rate",
    "host_identity_verified",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bathrooms_text",
    "bedrooms",
    "beds",
    "amenities_count",
    "price",
    "log_price",
    "minimum_nights",
    "maximum_nights",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "availability_rate",
    "number_of_reviews",
    "number_of_reviews_ltm",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "reviews_per_month",
    "has_reviews",
    "instant_bookable",
    "is_entire_home",
    "host_experience_days",
    "estimated_occupancy_l365d",
    "estimated_revenue_l365d",
    "high_price_listing"
]

available_columns = [col for col in selected_columns if col in df.columns]

clean_df = df[available_columns].copy()

clean_df.shape

(14091, 45)

In [53]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

clean_df.to_csv(DATA_PROCESSED / "cleaned_airbnb_athens.csv", index=False)
clean_df.to_csv(DATA_PROCESSED / "modeling_airbnb_athens.csv", index=False)

print("Saved cleaned datasets successfully.")

Saved cleaned datasets successfully.


In [54]:
clean_df.head()

,id,name,host_id,host_since,host_is_superhost,host_response_rate,host_acceptance_rate,host_identity_verified,neighbourhood_cleansed,latitude,...,review_scores_location,review_scores_value,reviews_per_month,has_reviews,instant_bookable,is_entire_home,host_experience_days,estimated_occupancy_l365d,estimated_revenue_l365d,high_price_listing
1,33945,Spacious Cosy aprtm very close to Metro!,146553,2010-06-17,0.0,100.0,84.0,1.0,ΑΓΙΟΣ ΝΙΚΟΛΑΟΣ,38.00673,...,4.64,4.81,0.49,1,0,1,5855.0,128,5760.0,0
2,49489,Ermou 44 - 3bdr apt in the heart of Athens,225612,2010-09-06,0.0,100.0,100.0,1.0,ΕΜΠΟΡΙΚΟ ΤΡΙΓΩΝΟ-ΠΛΑΚΑ,37.97670,...,4.89,4.69,0.83,1,1,1,5774.0,12,1920.0,1
3,60394,Cosy apartment! Great central Athens location!,290864,2010-11-18,1.0,NaN,100.0,1.0,ΣΤΑΔΙΟ,37.96738,...,4.92,4.92,0.29,1,0,1,5701.0,60,3000.0,0
5,154243,LUSCIOUS ROOF GARDEN IN THE CENTER!,741851,2011-06-25,0.0,60.0,77.0,1.0,ΜΟΥΣΕΙΟ-ΕΞΑΡΧΕΙΑ-ΝΕΑΠΟΛΗ,37.98574,...,4.60,4.66,1.30,1,0,1,5482.0,84,5880.0,0
6,155654,"Acropolis Cosy Apartment, Koukaki",712602,2011-06-17,0.0,100.0,NaN,1.0,ΑΚΡΟΠΟΛΗ,37.96828,...,4.95,4.71,0.47,1,0,1,5490.0,0,0.0,0


In [55]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14091 entries, 1 to 15582
Data columns (total 45 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   id                           14091 non-null  int64         
 1   name                         14091 non-null  object        
 2   host_id                      14091 non-null  int64         
 3   host_since                   14081 non-null  datetime64[ns]
 4   host_is_superhost            13208 non-null  float64       
 5   host_response_rate           12712 non-null  float64       
 6   host_acceptance_rate         13280 non-null  float64       
 7   host_identity_verified       14081 non-null  float64       
 8   neighbourhood_cleansed       14091 non-null  object        
 9   latitude                     14091 non-null  float64       
 10  longitude                    14091 non-null  float64       
 11  property_type                14091 non-null  o

In [56]:
clean_df.isna().mean().sort_values(ascending=False).head(20)

review_scores_checkin          0.135122
review_scores_location         0.135122
review_scores_communication    0.135122
review_scores_value            0.135122
reviews_per_month              0.135122
review_scores_rating           0.135122
review_scores_accuracy         0.135122
review_scores_cleanliness      0.135122
host_response_rate             0.097864
host_is_superhost              0.062664
host_acceptance_rate           0.057554
beds                           0.001916
bedrooms                       0.000923
host_identity_verified         0.000710
host_since                     0.000710
host_experience_days           0.000710
bathrooms_text                 0.000213
bathrooms                      0.000213
id                             0.000000
host_id                        0.000000
dtype: float64

In [57]:
clean_df.shape

(14091, 45)

We have managed to keep 45 out of 79 features.